# Figure 6 — Source Data Export

**Figure 6** examines the Hessian geometry of the GAN loss landscape and
its relationship to neural tuning curves.  Panel 6C shows an example
single-unit tuning curve sampled along Hessian eigenvectors.  Panel 6D
shows tuning heatmaps for preferred and non-preferred channels.  Panel 6E
summarises tuning shape statistics (peak sharpness, bandwidth) across the
population.

## Data requirements

> **Raw Hessian experiment data are required to run this notebook.**
> The `.pkl` result files are produced by the Hessian sampling experiments
> (see `analysis/hessian_geometry/`) and are not distributed with the
> repository.  Set the `MATROOT` environment variable to point to the
> directory containing those results, or assign the path directly to
> `matroot` in the cell below.
>
> ```bash
> export MATROOT=/path/to/Mat_Statistics
> ```

The exported source data files are written to `../source_data/Fig6/`.

In [ ]:
import os, sys, pickle, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from os.path import join

sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), ".."))
from neuro_data_analysis.neural_data_lib import (
    load_neural_data,
    extract_all_evol_trajectory_psth,
    pad_psth_traj,
    get_expstr,
    extract_natref_activation_array,
    extract_evol_activation_array,
)
from neuro_data_analysis.neural_data_utils import get_all_masks

In [ ]:
# Set the path to the raw Hessian experiment data.
# Override by setting the MATROOT environment variable before launching Jupyter.
matroot = os.environ.get("MATROOT", "")

# Directory containing Hessian sampling result tables
hessdir = join(matroot, "Hessian_Geometry")
outdir = "../source_data/Fig6"
os.makedirs(outdir, exist_ok=True)

## Figure 6C: Example tuning curve

Load the single-unit sampling data for experiment B07092020 and export the
response dataframe used to plot the example tuning curve.

In [ ]:
expid = "B07092020"

# Load pre-computed sampling result (produced by Hessian geometry pipeline)
sgtr_data = pickle.load(open(join(hessdir, f"{expid}_sgtr_resp_data.pkl"), "rb"))
sgtr_resp_df = sgtr_data["resp_df"]   # columns: eigvec_idx, alpha, resp_mean, resp_sem

sgtr_resp_df.to_csv(
    join(outdir, f"Figure6C_src_{expid}_sgtr_resp_df.csv"), index=False
)
pickle.dump(sgtr_resp_df, open(
    join(outdir, f"Figure6C_src_{expid}_sgtr_resp_df.pkl"), "wb"
))

print(f"sgtr_resp_df shape: {sgtr_resp_df.shape}")
print("Figure 6C source data saved.")

## Figure 6D: Tuning heatmaps

Load and export the population-average response heatmaps for preferred
channels (successful evolution) and non-preferred channels (failed evolution).

In [ ]:
# Preferred channel average response dataframe (successful evolution)
prefchan_df = pickle.load(open(join(hessdir, "prefchan_avgresp_df.pkl"), "rb"))
prefchan_df.to_csv(join(outdir, "Figure6D_src_prefchan_avgresp_df.csv"), index=False)
pickle.dump(prefchan_df, open(join(outdir, "Figure6D_src_prefchan_avgresp_df.pkl"), "wb"))

# Failed evolution channel average response dataframe
prefchan_fail_df = pickle.load(open(join(hessdir, "prefchan_avgresp_df_failevol.pkl"), "rb"))
prefchan_fail_df.to_csv(join(outdir, "Figure6D_src_prefchan_avgresp_df_failevol.csv"), index=False)
pickle.dump(prefchan_fail_df, open(join(outdir, "Figure6D_src_prefchan_avgresp_df_failevol.pkl"), "wb"))

print(f"prefchan_df shape:      {prefchan_df.shape}")
print(f"prefchan_fail_df shape: {prefchan_fail_df.shape}")
print("Figure 6D source data saved.")

## Figure 6E: Tuning shape statistics

Load and export the synopsis tables containing tuning-shape fitting
statistics (peak sharpness, bandwidth, R²) across the population.

In [ ]:
# Tuning shape fitting stats (Gaussian fit parameters per unit per eigvec)
fit_stats_pkl = pickle.load(open(join(hessdir, "tuning_shape_fitting_stats_synopsis.pkl"), "rb"))
sel_cols_fit = ["Expi", "eigvec_idx", "space", "R2", "peak", "sigma", "area"]
fit_stats_sel = fit_stats_pkl[sel_cols_fit] if all(c in fit_stats_pkl.columns for c in sel_cols_fit) \
                else fit_stats_pkl
fit_stats_sel.to_csv(
    join(outdir, "Figure6E_src_tuning_shape_fitting_stats_synopsis_selcolumn.csv"), index=False
)
pickle.dump(fit_stats_sel, open(
    join(outdir, "Figure6E_src_tuning_shape_fitting_stats_synopsis_sel.pkl"), "wb"
))

# Broader tuning statistics synopsis
tuning_stats_pkl = pickle.load(open(join(hessdir, "tuning_stats_synopsis.pkl"), "rb"))
sel_cols_tun = ["Expi", "space", "area", "DeePSim_peak", "BigGAN_peak", "DeePSim_sigma", "BigGAN_sigma"]
tuning_stats_sel = tuning_stats_pkl[sel_cols_tun] if all(c in tuning_stats_pkl.columns for c in sel_cols_tun) \
                   else tuning_stats_pkl
tuning_stats_sel.to_csv(
    join(outdir, "Figure6E_src_tuning_stats_synopsis_selcolumn.csv"), index=False
)
pickle.dump(tuning_stats_sel, open(
    join(outdir, "Figure6E_src_tuning_stats_synopsis_selcolumn.pkl"), "wb"
))

print(f"fit_stats_sel shape:    {fit_stats_sel.shape}")
print(f"tuning_stats_sel shape: {tuning_stats_sel.shape}")
print("Figure 6E source data saved to", outdir)